# CNN 실습 001 — MNIST 손글씨 숫자 인식

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/cnn_001_mnist_handwriting.ipynb)

**CNN(Convolutional Neural Network, 합성곱 신경망)**으로 손글씨 숫자(0~9)를 인식하는 실습입니다.

### 이 노트북에서 배울 것

| 순서 | 내용 |
|------|------|
| 1 | MNIST 데이터셋 이해 (28×28 흑백 이미지) |
| 2 | CNN의 핵심 구성요소: Conv2D, MaxPooling, Flatten, Dense |
| 3 | 합성곱(Convolution) 연산이 이미지에서 특징을 추출하는 원리 |
| 4 | 모델 학습, 평가, 예측 |
| 5 | 필터(커널) 시각화 — CNN이 실제로 무엇을 보는지 확인 |

### CNN이 뭔가요? (비유)

> 사람이 숫자 "7"을 인식할 때:
> 1. 먼저 **부분적 특징**을 봅니다 — "위에 가로선이 있네", "대각선이 있네"
> 2. 그 특징들을 **조합**해서 "이건 7이다"라고 판단합니다
>
> CNN도 똑같습니다:
> 1. **Conv2D** (합성곱층): 이미지의 부분적 특징(가로선, 세로선, 곡선 등)을 감지
> 2. **MaxPooling** (풀링층): 중요한 특징만 남기고 크기를 줄임
> 3. **Dense** (완전연결층): 추출된 특징들을 조합해서 "이건 숫자 7"이라고 판단

---
## 1. 환경 설정 & 라이브러리 임포트

In [ ]:
# ============================================================
# 라이브러리 임포트
# ============================================================

# tensorflow: 구글이 만든 딥러닝 프레임워크
import tensorflow as tf

# keras: tensorflow 안에 포함된 고수준 딥러닝 API
# - 복잡한 텐서 연산을 간단한 함수 호출로 사용 가능
from tensorflow import keras

# Sequential: 레이어를 순서대로 쌓는 가장 기본적인 모델 구조
from tensorflow.keras.models import Sequential

# Input: 모델 입력 형태를 명시적으로 정의 (경고 없이 깔끔하게)
# Conv2D: 2D 합성곱층 — 이미지에서 특징(패턴)을 추출하는 핵심 레이어
# MaxPooling2D: 최대 풀링층 — 특징맵의 크기를 줄여 계산량 감소 + 위치 불변성
# Flatten: 2D 특징맵을 1D 벡터로 펼침 (Dense 레이어 입력용)
# Dense: 완전연결층 — 추출된 특징을 조합하여 최종 분류
# Dropout: 무작위로 뉴런을 꺼서 과적합 방지
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
)

# numpy: 수치 계산 라이브러리
import numpy as np

# matplotlib: 그래프/이미지 시각화
import matplotlib.pyplot as plt

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 가능: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 2. MNIST 데이터셋 로드 & 탐색

### MNIST란?
- **M**odified **N**ational **I**nstitute of **S**tandards and **T**echnology
- 0~9 손글씨 숫자 이미지 **70,000장** (학습 60,000 + 테스트 10,000)
- 각 이미지: **28 × 28 픽셀**, 흑백 (0=검정, 255=흰색)
- 딥러닝의 "Hello World" — 가장 먼저 해보는 실습

```
이미지 하나의 구조:
┌──────────────────────┐
│  28 픽셀 (가로)       │
│ ┌──┬──┬──┬─ ... ─┬──┐│  28 픽셀
│ │0 │0 │12│       │0 ││  (세로)
│ ├──┼──┼──┤       ├──┤│
│ │0 │45│200       │0 ││  각 칸 = 1 픽셀
│ │  │  │  │ ...   │  ││  값: 0(검정) ~ 255(흰색)
│ └──┴──┴──┴───────┴──┘│
└──────────────────────┘
```

In [ ]:
# ============================================================
# MNIST 데이터셋 로드
# ============================================================

# keras.datasets.mnist: MNIST 데이터셋을 자동 다운로드
# load_data() 반환값:
#   (X_train, y_train): 학습용 이미지 60,000장 + 정답 레이블
#   (X_test, y_test):   테스트용 이미지 10,000장 + 정답 레이블
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# 데이터 형태 확인
print("=" * 50)
print("MNIST 데이터셋 정보")
print("=" * 50)
print(f"학습 이미지: {X_train.shape}")  # (60000, 28, 28)
print(f"  → {X_train.shape[0]}장, 각 {X_train.shape[1]}×{X_train.shape[2]} 픽셀")
print(f"학습 레이블: {y_train.shape}")  # (60000,)
print(f"테스트 이미지: {X_test.shape}") # (10000, 28, 28)
print(f"테스트 레이블: {y_test.shape}") # (10000,)
print(f"\n픽셀 값 범위: {X_train.min()} ~ {X_train.max()}")  # 0 ~ 255
print(f"레이블 종류: {sorted(set(y_train))}")  # [0, 1, 2, ..., 9]

In [ ]:
# ============================================================
# 데이터 시각화 — 실제 손글씨 이미지 확인
# ============================================================

# 학습 데이터에서 처음 16장을 4×4 격자로 출력
fig, axes = plt.subplots(2, 8, figsize=(14, 4))

for i, ax in enumerate(axes.flat):
    # imshow: 2D 배열을 이미지로 표시
    # cmap='gray': 흑백 컬러맵 (0=검정, 255=흰색)
    ax.imshow(X_train[i], cmap='gray')

    # 이미지 위에 정답 레이블 표시
    ax.set_title(f"정답: {y_train[i]}", fontsize=11)

    # 축 눈금 숨기기 (이미지에 불필요)
    ax.axis('off')

plt.suptitle("MNIST 손글씨 숫자 샘플", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 이미지 1장의 픽셀 값 직접 확인
# ============================================================

# 숫자 하나를 골라서 실제 픽셀 값을 봅니다
sample_idx = 0  # 첫 번째 이미지
sample_img = X_train[sample_idx]

print(f"이미지 #{sample_idx}의 정답: {y_train[sample_idx]}")
print(f"이미지 크기: {sample_img.shape}")
print(f"\n중앙 부분(10~18행, 10~18열)의 픽셀 값:")
print(sample_img[10:18, 10:18])
print("\n→ 0에 가까우면 검정(배경), 255에 가까우면 흰색(글자)")

---
## 3. 데이터 전처리

CNN에 넣기 전에 두 가지 전처리가 필요합니다:

### 3.1 정규화 (Normalization)
- 픽셀 값을 **0~255** → **0.0~1.0** 으로 변환
- 왜? 값이 크면 가중치 업데이트가 불안정해짐
- 비유: 시험 점수를 100점 만점 → 1점 만점으로 바꾸는 것

### 3.2 차원 추가 (채널 차원)
- 원본: `(60000, 28, 28)` → 변환: `(60000, 28, 28, 1)`
- 마지막 `1`은 **채널 수** (흑백=1, 컬러RGB=3)
- Conv2D는 4D 입력을 요구: `(배치, 높이, 너비, 채널)`

```
흑백 이미지:  (28, 28, 1)  ← 채널 1개
컬러 이미지:  (28, 28, 3)  ← R, G, B 채널 3개
```

In [ ]:
# ============================================================
# 정규화: 0~255 → 0.0~1.0
# ============================================================

# astype('float32'): 정수(int) → 실수(float)로 변환
#   - 정수 나눗셈은 소수점이 잘리므로, 먼저 실수로 바꿔야 함
#   - float32: 32비트 부동소수점 (GPU 연산에 최적화된 타입)
# / 255.0: 최대값(255)으로 나눠서 0~1 범위로 정규화
#   - 예: 128 / 255.0 = 0.502, 255 / 255.0 = 1.0, 0 / 255.0 = 0.0
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"정규화 후 픽셀 값 범위: {X_train.min():.1f} ~ {X_train.max():.1f}")

# ============================================================
# 채널 차원 추가: (28, 28) → (28, 28, 1)
# ============================================================

# np.expand_dims(배열, axis): 지정한 위치에 크기 1인 차원을 추가
# axis=-1: 맨 마지막에 추가
# (60000, 28, 28) → (60000, 28, 28, 1)
#                                    ↑ 채널 차원 (흑백이므로 1)
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print(f"\n차원 추가 후:")
print(f"  X_train: {X_train.shape}")  # (60000, 28, 28, 1)
print(f"  X_test:  {X_test.shape}")   # (10000, 28, 28, 1)
print(f"  → (샘플 수, 높이, 너비, 채널)")

---
## 4. CNN 모델 구성

### 합성곱(Convolution)이란?

작은 필터(커널)를 이미지 위에서 **슬라이딩**하면서 특징을 추출하는 연산입니다.

```
입력 이미지 (5×5)          필터/커널 (3×3)       특징맵 (출력)
┌──┬──┬──┬──┬──┐          ┌──┬──┬──┐
│1 │0 │1 │0 │1 │          │1 │0 │1 │           쉽게 말하면:
├──┼──┼──┼──┼──┤    *     ├──┼──┼──┤    =      필터가 이미지 위를
│0 │1 │0 │1 │0 │          │0 │1 │0 │           한 칸씩 이동하면서
├──┼──┼──┼──┼──┤          ├──┼──┼──┤           "이 패턴이 있나?"
│1 │0 │1 │0 │1 │          │1 │0 │1 │           확인하는 것
├──┼──┼──┼──┼──┤          └──┴──┴──┘
│0 │1 │0 │1 │0 │
├──┼──┼──┼──┼──┤
│1 │0 │1 │0 │1 │
└──┴──┴──┴──┴──┘

필터가 감지하는 것 (예시):
  ┌───┐  ┌───┐  ┌───┐  ┌───┐
  │━━━│  │ ┃ │  │ ╱ │  │╲  │  ← 가로선, 세로선, 대각선 등
  │   │  │ ┃ │  │╱  │  │ ╲ │    CNN이 학습하면서 자동으로
  │   │  │ ┃ │  │   │  │  ╲│    유용한 필터를 찾아냄
  └───┘  └───┘  └───┘  └───┘
```

### 합성곱 연산 과정 (숫자 예시)

```
이미지 3×3 영역       필터 3×3
┌──┬──┬──┐          ┌──┬──┬──┐
│1 │2 │3 │    ×     │1 │0 │-1│    = 1×1 + 2×0 + 3×(-1)
├──┼──┼──┤          ├──┼──┼──┤      + 0×1 + 1×0 + 2×(-1)
│0 │1 │2 │          │1 │0 │-1│      + 1×1 + 0×0 + 1×(-1)
├──┼──┼──┤          ├──┼──┼──┤
│1 │0 │1 │          │1 │0 │-1│    = 1 + 0 - 3 + 0 + 0 - 2 + 1 + 0 - 1
└──┴──┴──┘          └──┴──┴──┘    = -4

→ 원소별로 곱한 뒤 모두 더함 = 하나의 숫자 출력
→ 필터를 한 칸씩 이동하면서 반복 = 특징맵(Feature Map) 완성
```

### MaxPooling이란?

```
특징맵 (4×4)              MaxPool 2×2           출력 (2×2)
┌──┬──┬──┬──┐                                  ┌──┬──┐
│1 │3 │2 │1 │     2×2 영역에서               │4 │3 │
├──┼──┼──┼──┤     가장 큰 값만               ├──┼──┤
│4 │2 │1 │0 │     남기고 나머지 버림          │7 │5 │
├──┼──┼──┼──┤     → 크기가 절반으로 줄어듦     └──┴──┘
│7 │1 │5 │3 │
├──┼──┼──┼──┤     왜? 1) 계산량 감소
│2 │0 │4 │1 │         2) 특징의 위치가 조금 달라도 감지 가능
└──┴──┴──┴──┘            (위치 불변성)
```

### 전체 모델 구조

```
입력 (28×28×1)
    ↓
Conv2D(32, 3×3) + ReLU    ← 32개 필터로 특징 추출 → (26×26×32)
    ↓
MaxPooling2D(2×2)          ← 크기 절반으로 축소 → (13×13×32)
    ↓
Conv2D(64, 3×3) + ReLU    ← 64개 필터로 고수준 특징 추출 → (11×11×64)
    ↓
MaxPooling2D(2×2)          ← 크기 절반으로 축소 → (5×5×64)
    ↓
Flatten                    ← 2D → 1D로 펼침 → (1600)
    ↓
Dropout(0.5)               ← 50% 뉴런 무작위 비활성화 (과적합 방지)
    ↓
Dense(128) + ReLU          ← 특징 조합 → (128)
    ↓
Dense(10) + Softmax        ← 10개 클래스 확률 출력 → (10)
    ↓
출력: [0.01, 0.01, 0.02, 0.90, 0.01, ...] ← "이건 숫자 3"
```

In [ ]:
# ============================================================
# CNN 모델 생성
# ============================================================

model = Sequential()

# ── 입력층 ───────────────────────────────────────────────
# Input(shape=(28, 28, 1)): 입력 이미지의 형태를 정의
# - 28: 이미지 높이 (픽셀)
# - 28: 이미지 너비 (픽셀)
# - 1:  채널 수 (흑백=1, 컬러RGB=3)
model.add(Input(shape=(28, 28, 1)))

# ── 첫 번째 합성곱 블록 ──────────────────────────────────

# Conv2D(32, (3, 3), activation='relu')
# - 32: 필터(커널) 개수
#   → 32개의 서로 다른 필터가 각각 다른 특징을 감지
#   → 필터 1: 가로선 감지, 필터 2: 세로선 감지, 필터 3: 곡선 감지...
#   → 출력: 32장의 특징맵(feature map)
#
# - (3, 3): 필터 크기 (3×3 픽셀)
#   → 한 번에 3×3=9개 픽셀 영역을 봄
#   → 작은 필터 = 세밀한 특징 감지 (선, 모서리 등)
#   → 큰 필터(5×5, 7×7) = 넓은 영역의 특징 감지
#
# - activation='relu': 활성화 함수
#   → f(x) = max(0, x)
#   → 합성곱 결과에서 음수를 0으로 만듦
#   → "이 특징이 있으면 양수, 없으면 0" 으로 해석 가능
#
# - padding='valid' (기본값): 패딩 없음
#   → 28×28 이미지에 3×3 필터 적용 → 출력 크기: 26×26
#   → 공식: 출력 크기 = 입력 크기 - 필터 크기 + 1 = 28 - 3 + 1 = 26
#
# 학습 파라미터 수:
#   - 필터 1개 = 3×3×1(입력 채널) + 1(편향) = 10개
#   - 필터 32개 = 10 × 32 = 320개
model.add(Conv2D(32, (3, 3), activation='relu'))

# MaxPooling2D((2, 2)): 최대 풀링
# - (2, 2): 풀링 윈도우 크기
#   → 2×2 영역에서 가장 큰 값 1개만 남김
#   → 출력 크기: 26×26 → 13×13 (절반으로 축소)
# - 효과:
#   1. 계산량 감소 (파라미터 수는 0 — 학습할 것이 없음)
#   2. 위치 불변성: 특징이 약간 이동해도 동일하게 감지
#      → 숫자 "7"이 이미지 왼쪽에 있든 오른쪽에 있든 인식 가능
model.add(MaxPooling2D((2, 2)))

# ── 두 번째 합성곱 블록 ──────────────────────────────────

# Conv2D(64, (3, 3), activation='relu')
# - 64: 더 많은 필터 → 더 복잡한 특징 감지
# - 첫 번째 층이 감지한 "선/모서리" 특징을 조합하여
#   "원", "ㄱ자 꺾임", "곡선" 같은 고수준 특징을 감지
# - 입력: 13×13×32 → 출력: 11×11×64
#
# 학습 파라미터 수:
#   - 필터 1개 = 3×3×32(입력 채널) + 1(편향) = 289개
#   - 필터 64개 = 289 × 64 = 18,496개
#
# 비유: 1층(Conv2D-32)은 "획"을 보고, 2층(Conv2D-64)은 "글자 부분"을 봄
model.add(Conv2D(64, (3, 3), activation='relu'))

# MaxPooling2D: 11×11 → 5×5
model.add(MaxPooling2D((2, 2)))

# ── 분류기 (Classifier) ──────────────────────────────────

# Flatten(): 2D 특징맵 → 1D 벡터로 펼침
# - 입력: (5, 5, 64) = 5×5×64 = 1,600개 값
# - 출력: (1600,) — 1차원 벡터
# - 왜? Dense 레이어는 1D 입력만 받을 수 있음
# - 비유: 2D 지도를 한 줄로 쭉 펼치는 것
model.add(Flatten())

# Dropout(0.5): 드롭아웃 — 과적합 방지
# - 0.5: 학습 시 뉴런의 50%를 무작위로 비활성화
# - 왜? 특정 뉴런에만 의존하지 않고 여러 뉴런이 골고루 학습하도록
# - 비유: 팀 프로젝트에서 매번 다른 팀원 조합으로 연습
#   → 누가 빠져도 잘 할 수 있는 팀이 됨
# - 추론(예측) 시에는 자동으로 꺼짐 (모든 뉴런 사용)
model.add(Dropout(0.5))

# Dense(128, activation='relu'): 완전연결층
# - 128: 뉴런 수
# - 추출된 1,600개 특징을 128개로 압축/조합
# - 학습 파라미터: 1,600 × 128 + 128(편향) = 204,928개
model.add(Dense(128, activation='relu'))

# Dense(10, activation='softmax'): 출력층
# - 10: 숫자 0~9 = 10개 클래스
# - softmax: 출력을 확률 분포로 변환 (합 = 1)
#   → 예: [0.01, 0.01, 0.02, 0.90, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
#   → "이 이미지가 숫자 3일 확률 90%"
model.add(Dense(10, activation='softmax'))

In [ ]:
# ============================================================
# 모델 구조 확인
# ============================================================

# model.summary(): 모델의 전체 구조를 표로 출력
# - Layer: 레이어 종류
# - Output Shape: 각 레이어의 출력 크기
#   - None = 배치 크기 (실행 시 결정)
# - Param #: 학습 가능한 파라미터 수
model.summary()

---
## 5. 모델 컴파일 & 학습

### 학습 과정 요약

```
1. 이미지 배치(128장)를 모델에 입력
2. 순전파: 이미지 → Conv → Pool → Conv → Pool → Flatten → Dense → 예측
3. 손실 계산: 예측값과 정답의 차이 (sparse_categorical_crossentropy)
4. 역전파: 각 필터/가중치의 기울기 계산
5. 가중치 업데이트: Adam 옵티마이저가 필터 값과 Dense 가중치를 수정
6. 1~5를 반복 → 1 epoch = 전체 학습 데이터 1회 통과
```

In [ ]:
# ============================================================
# 모델 컴파일
# ============================================================

model.compile(
    # optimizer='adam': Adam 옵티마이저
    # - 학습률을 파라미터별로 자동 조절
    # - "잘 모르겠으면 Adam" — 가장 범용적인 옵티마이저
    optimizer='adam',

    # loss='sparse_categorical_crossentropy':
    # - 다중 클래스 분류용 손실 함수
    # - 'sparse_': 정수 레이블(0,1,2,...,9)을 바로 사용 가능
    #   → 원-핫 인코딩 불필요! (categorical_crossentropy는 원-핫 필요)
    # - 정답 클래스의 예측 확률이 높을수록 손실이 낮아짐
    loss='sparse_categorical_crossentropy',

    # metrics=['accuracy']: 학습 중 정확도 추적
    # - 정답을 맞춘 비율 = 맞춘 수 / 전체 수
    metrics=['accuracy']
)

print("모델 컴파일 완료!")

In [ ]:
# ============================================================
# 모델 학습
# ============================================================

history = model.fit(
    X_train,         # 학습 이미지 (60,000장)
    y_train,         # 학습 정답 레이블 (0~9 정수)

    # epochs=10: 전체 데이터를 10번 반복 학습
    # - 1 epoch = 60,000장 모두 한 번씩 학습
    # - 매 epoch마다 정확도가 올라가는지 확인
    epochs=10,

    # batch_size=128: 한 번에 128장씩 묶어서 학습
    # - 60,000 ÷ 128 ≈ 469번 가중치 업데이트/epoch
    # - 작으면: 업데이트 빈번 → 느리지만 세밀한 학습
    # - 크면: 업데이트 적음 → 빠르지만 거친 학습
    # - 128은 GPU 메모리와 학습 효율의 균형점
    batch_size=128,

    # validation_split=0.1: 학습 데이터의 10%를 검증용으로 분리
    # - 학습: 54,000장 / 검증: 6,000장
    # - 왜? 과적합 여부를 학습 중에 모니터링
    # - val_accuracy가 올라가다 내려가기 시작하면 = 과적합 시작
    validation_split=0.1,

    # verbose=1: 학습 진행 상황을 프로그레스바로 출력
    verbose=1
)

print("\n학습 완료!")

---
## 6. 학습 결과 시각화

In [ ]:
# ============================================================
# 정확도 & 손실 그래프
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── 정확도 그래프 ────────────────────────────────────────
# history.history['accuracy']: 학습 데이터 정확도 (에포크별)
# history.history['val_accuracy']: 검증 데이터 정확도 (에포크별)
ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training & Validation Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── 손실 그래프 ──────────────────────────────────────────
ax2.plot(history.history['loss'], label='Train Loss', linewidth=2, color='tab:red')
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='tab:orange', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training & Validation Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 그래프 읽는 법:
# - Train과 Val이 함께 올라가면 → 정상 학습
# - Train은 올라가는데 Val이 내려가면 → 과적합
# - 둘 다 안 올라가면 → 학습 부족 (에포크 더 필요하거나 모델이 너무 단순)

---
## 7. 테스트 데이터로 최종 평가

In [ ]:
# ============================================================
# 테스트 데이터 평가
# ============================================================

# model.evaluate(): 테스트 데이터에 대한 손실과 정확도 계산
# - 이 데이터는 학습에 한 번도 사용되지 않은 "처음 보는 데이터"
# - 이 정확도가 모델의 진짜 실력
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"테스트 데이터 정확도: {test_acc:.4f} ({test_acc:.1%})")
print(f"테스트 데이터 손실:   {test_loss:.4f}")
print(f"\n→ 10,000장의 처음 보는 손글씨 중 {int(test_acc * 10000)}장을 정확히 인식!")

---
## 8. 예측 결과 시각화 — 맞춘 것 & 틀린 것

In [ ]:
# ============================================================
# 예측 수행
# ============================================================

# model.predict(): 테스트 이미지에 대한 예측 확률 계산
# - 반환값: (10000, 10) 배열
#   - 각 행: 이미지 1장에 대한 10개 클래스 확률
#   - 예: [0.01, 0.01, 0.02, 0.90, 0.01, ...] → "숫자 3일 확률 90%"
predictions = model.predict(X_test, verbose=0)

# argmax: 가장 높은 확률의 인덱스 = 예측 숫자
pred_labels = np.argmax(predictions, axis=1)

print(f"예측 결과 shape: {predictions.shape}")
print(f"\n첫 5개 이미지 예측:")
for i in range(5):
    conf = predictions[i][pred_labels[i]]  # 예측 확률
    correct = "O" if pred_labels[i] == y_test[i] else "X"
    print(f"  [{correct}] 정답: {y_test[i]}, 예측: {pred_labels[i]} (확률: {conf:.1%})")

In [ ]:
# ============================================================
# 맞춘 예시 & 틀린 예시 시각화
# ============================================================

# 맞춘 것과 틀린 것의 인덱스 분리
correct_idx = np.where(pred_labels == y_test)[0]   # 정답 맞춘 인덱스
wrong_idx = np.where(pred_labels != y_test)[0]     # 틀린 인덱스

print(f"맞춘 수: {len(correct_idx)}, 틀린 수: {len(wrong_idx)}")

fig, axes = plt.subplots(2, 8, figsize=(16, 5))

# 윗줄: 맞춘 것 8개
for i in range(8):
    idx = correct_idx[i]
    axes[0, i].imshow(X_test[idx].squeeze(), cmap='gray')
    axes[0, i].set_title(f"O {pred_labels[idx]}", color='green', fontsize=12, fontweight='bold')
    axes[0, i].axis('off')

# 아랫줄: 틀린 것 8개 (있는 만큼만)
n_wrong_show = min(8, len(wrong_idx))
for i in range(n_wrong_show):
    idx = wrong_idx[i]
    axes[1, i].imshow(X_test[idx].squeeze(), cmap='gray')
    axes[1, i].set_title(f"X {pred_labels[idx]}(정답:{y_test[idx]})", color='red', fontsize=10, fontweight='bold')
    axes[1, i].axis('off')

# 빈 칸 숨기기
for i in range(n_wrong_show, 8):
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correct', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Wrong', fontsize=12, fontweight='bold')
plt.suptitle('CNN 예측 결과', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. CNN이 보는 것 — 필터 & 특징맵 시각화

CNN이 학습한 **필터(커널)**와 이미지에 필터를 적용한 **특징맵(feature map)**을 직접 확인합니다.

이것이 CNN의 강점: **어떤 특징을 보고 판단했는지** 시각적으로 확인 가능!

In [ ]:
# ============================================================
# 첫 번째 Conv2D 레이어의 필터(커널) 시각화
# ============================================================

# model.layers: 모델의 모든 레이어 리스트
# Conv2D 레이어 찾기
conv_layers = [layer for layer in model.layers if 'conv2d' in layer.name]
first_conv = conv_layers[0]

# get_weights(): [가중치(필터), 편향] 반환
# - 가중치 shape: (3, 3, 1, 32) = (높이, 너비, 입력채널, 필터수)
filters, biases = first_conv.get_weights()
print(f"첫 번째 Conv2D 필터 shape: {filters.shape}")
print(f"  → 3×3 크기의 필터 {filters.shape[3]}개")

# 필터 16개를 4×4 격자로 시각화
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    if i < 16:
        # filters[:, :, 0, i]: i번째 필터의 3×3 가중치
        ax.imshow(filters[:, :, 0, i], cmap='gray')
        ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')

plt.suptitle('CNN이 학습한 3×3 필터들 (밝으면 양수, 어두우면 음수)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("→ 각 필터가 서로 다른 패턴(가로선, 세로선, 대각선 등)을 감지하도록 학습됨")

In [ ]:
# ============================================================
# 특징맵(Feature Map) 시각화 — CNN이 이미지에서 실제로 보는 것
# ============================================================

# 테스트 이미지 1장을 선택
sample = X_test[0:1]  # (1, 28, 28, 1) — 배치 차원 유지

# 중간 레이어의 출력을 추출하는 모델 생성
# - 첫 번째 Conv2D의 출력 = 32장의 특징맵
feature_model = keras.Model(
    inputs=model.input,
    outputs=conv_layers[0].output  # 첫 번째 Conv2D 출력
)

# 특징맵 추출
feature_maps = feature_model.predict(sample, verbose=0)
print(f"특징맵 shape: {feature_maps.shape}")  # (1, 26, 26, 32)

# 원본 이미지 + 특징맵 16개 시각화
fig, axes = plt.subplots(2, 9, figsize=(16, 4))

# 원본 이미지
axes[0, 0].imshow(sample.squeeze(), cmap='gray')
axes[0, 0].set_title(f'Original\n(label:{y_test[0]})', fontsize=9)
axes[0, 0].axis('off')

# 특징맵 16개
for i in range(16):
    row = (i + 1) // 9
    col = (i + 1) % 9
    axes[row, col].imshow(feature_maps[0, :, :, i], cmap='viridis')
    axes[row, col].set_title(f'Map {i}', fontsize=8)
    axes[row, col].axis('off')

# 빈 칸 숨기기
for i in range(16 + 1, 18):
    axes[i // 9, i % 9].axis('off')

plt.suptitle('CNN 첫 번째 Conv2D 출력 — 각 필터가 감지한 특징', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("→ 각 특징맵은 서로 다른 패턴(가로선, 세로선, 곡선 등)이 어디에 있는지 보여줌")
print("→ 밝은 부분 = 해당 필터가 강하게 반응한 영역")

---
## 10. 내가 직접 그린 숫자 예측 (Colab에서 실행)

Colab에서는 마우스로 숫자를 직접 그려서 CNN이 인식하는지 테스트할 수 있습니다!

In [ ]:
# ============================================================
# 랜덤 테스트 이미지로 예측 (Colab/로컬 모두 실행 가능)
# ============================================================

# 테스트 데이터에서 랜덤 10장 선택
random_indices = np.random.choice(len(X_test), 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, idx in enumerate(random_indices):
    ax = axes[i // 5, i % 5]

    # 이미지 표시
    ax.imshow(X_test[idx].squeeze(), cmap='gray')

    # 예측
    pred_prob = predictions[idx]
    pred_label = np.argmax(pred_prob)
    confidence = pred_prob[pred_label]
    true_label = y_test[idx]

    # 맞으면 초록, 틀리면 빨강
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"예측:{pred_label} ({confidence:.0%})\n정답:{true_label}",
                 color=color, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('랜덤 테스트 이미지 예측 결과', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 예측 확률 분포 시각화 — 모델이 얼마나 확신하는지
# ============================================================

# 틀린 예시 하나를 골라서 확률 분포를 봅니다
if len(wrong_idx) > 0:
    idx = wrong_idx[0]  # 첫 번째 오답
    title = f"오답 예시 (정답: {y_test[idx]})"
else:
    idx = 0
    title = f"예시 (정답: {y_test[idx]})"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# 이미지
ax1.imshow(X_test[idx].squeeze(), cmap='gray')
ax1.set_title(title, fontsize=12)
ax1.axis('off')

# 확률 바 차트
probs = predictions[idx]
colors = ['tab:red' if i == np.argmax(probs) else 'tab:blue' for i in range(10)]
ax2.barh(range(10), probs, color=colors)
ax2.set_yticks(range(10))
ax2.set_yticklabels([str(i) for i in range(10)])
ax2.set_xlabel('Probability')
ax2.set_title(f'CNN 예측 확률 (빨강 = 최종 예측: {np.argmax(probs)})', fontsize=11)
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
## 11. 핵심 개념 정리

### CNN 구성 요소 요약

| 레이어 | 역할 | 비유 |
|--------|------|------|
| **Conv2D** | 필터로 이미지에서 특징(패턴) 추출 | 돋보기로 부분을 하나씩 살펴보기 |
| **MaxPooling2D** | 중요한 특징만 남기고 크기 축소 | 요약 노트 만들기 |
| **Flatten** | 2D 특징맵을 1D 벡터로 변환 | 지도를 한 줄로 펼치기 |
| **Dropout** | 무작위로 뉴런 비활성화 (과적합 방지) | 팀원 돌아가며 빠지기 훈련 |
| **Dense** | 추출된 특징을 조합하여 분류 | 종합 판단 |
| **Softmax** | 출력을 확률 분포로 변환 | "이건 7일 확률 95%" |

### CNN vs 일반 신경망(MLP)

| | MLP (Dense만 사용) | CNN |
|---|------------------|-----|
| 이미지 처리 | 픽셀을 1D로 펼침 (784개) | 2D 구조 유지 |
| 특징 추출 | 수동 (사람이 설계) | 자동 (필터가 학습) |
| 위치 불변성 | 없음 | MaxPooling으로 확보 |
| 파라미터 수 | 매우 많음 | 적음 (가중치 공유) |
| 이미지 인식 | 약함 | 강함 |

### CNN의 핵심 아이디어: 가중치 공유 (Weight Sharing)

> 하나의 3×3 필터(9개 가중치)를 이미지 전체에 **재사용**합니다.
> MLP라면 28×28=784개 픽셀 각각에 별도 가중치가 필요하지만,
> CNN은 9개 가중치로 전체 이미지를 스캔합니다.
> → 파라미터가 훨씬 적음 → 과적합 위험 감소 → 이미지에 강함